# 第8章 数据库（二）：DBMS 与数据库管理
# Chapter 8 Databases (Part 2): DBMS & Database Management

---

**Cambridge International AS & A Level Computer Science (9618)**

> "A database management system (DBMS) is to data what an operating system is to hardware — an essential layer of abstraction."

上一节我们了解了数据库的基本概念和关系模型。这一节，我们将学习**数据库管理系统（DBMS）**——它是用户与物理数据之间的"桥梁"。

**本节学习目标：**
- 理解 DBMS 是什么，为什么需要它
- 掌握 DBMS 的核心功能（数据字典、安全性、并发控制等）
- 了解 DBMS 的工具（查询处理器、开发接口等）
- 实际使用 Python 的 sqlite3 模块操作数据库

## 8.2.1 什么是 DBMS？ | What is a DBMS?

**DBMS（Database Management System，数据库管理系统）** 是一个软件系统，它充当用户/应用程序与物理数据库之间的**中间层（intermediary layer）**。

### 三层架构 | Three-Level Architecture

```
用户/应用程序 (Users / Applications)
        ↕
   ╔══════════════════╗
   ║      DBMS        ║  ← 软件层（Software Layer）
   ║  ┌────────────┐  ║
   ║  │ Query      │  ║  - 接收用户请求
   ║  │ Processor  │  ║  - 验证权限
   ║  ├────────────┤  ║  - 优化查询
   ║  │ Security   │  ║  - 执行操作
   ║  │ Manager    │  ║  - 返回结果
   ║  ├────────────┤  ║
   ║  │ Storage    │  ║
   ║  │ Manager    │  ║
   ║  └────────────┘  ║
   ╚══════════════════╝
        ↕
   物理数据库 (Physical Database on Disk)
```

> **类比：** DBMS 就像一个**超级图书管理员**。
> - 你不需要自己去书库找书（不需要直接操作磁盘文件）
> - 你只需告诉管理员你要什么（写 SQL 查询）
> - 管理员帮你找到书、检查你有没有权限借、记录借阅信息（DBMS 处理一切）
> - 如果两个人同时想借同一本书，管理员会协调（并发控制）

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'simhei' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = 'C:/Windows/Fonts/simhei.ttf'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import matplotlib.patches as mpatches


fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('DBMS Core Features Overview', fontsize=16, fontweight='bold')

# Central DBMS box
central = mpatches.FancyBboxPatch((5, 3.5), 4, 2, boxstyle='round,pad=0.2',
                                   facecolor='#d5f5e3', edgecolor='#27ae60', linewidth=3)
ax.add_patch(central)
ax.text(7, 4.5, 'DBMS', ha='center', va='center', fontsize=20, fontweight='bold', color='#27ae60')

# Feature boxes around the center
features = [
    ('Data Dictionary\n(metadata)', 1.5, 7, '#aed6f1', '#2980b9'),
    ('Data Integrity\n(constraints)', 5.5, 7, '#f9e79f', '#f39c12'),
    ('Security &\nAccess Control', 10, 7, '#f5b7b1', '#e74c3c'),
    ('Concurrent\nAccess Control', 1.5, 1, '#d7bde2', '#8e44ad'),
    ('Query\nProcessor', 5.5, 1, '#a9dfbf', '#27ae60'),
    ('Backup &\nRecovery', 10, 1, '#fadbd8', '#c0392b'),
    ('Data Modeling\nTools', 12.5, 4, '#d5dbdb', '#566573'),
    ('Logical\nSchema', 0.5, 4, '#abebc6', '#1e8449'),
]

for label, x, y, fcolor, ecolor in features:
    rect = mpatches.FancyBboxPatch((x - 1.2, y - 0.6), 2.4, 1.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor=fcolor, edgecolor=ecolor, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')
    # Draw line to center
    cx, cy = 7, 4.5
    ax.annotate('', xy=(cx + (x-cx)*0.35, cy + (y-cy)*0.35),
                xytext=(x + (cx-x)*0.35, y + (cy-y)*0.35),
                arrowprops=dict(arrowstyle='->', color=ecolor, lw=1.5))

plt.tight_layout()
plt.savefig('dbms_features.png', dpi=100, bbox_inches='tight')
plt.show()
print('DBMS provides all these features as a unified software layer.')


## 8.2.2 DBMS 的核心功能 | Core Features of DBMS

### 1. 数据字典（Data Dictionary）

数据字典是**关于数据的数据（metadata）**——它记录了数据库本身的结构信息。

> **类比：** 如果数据库是一本书，那么数据字典就是这本书的**目录和前言**——告诉你这本书有哪些章节、每章多少页、用什么语言写的。

数据字典记录了：
- 所有表的名称
- 每个表有哪些字段（列名、数据类型、约束）
- 主键和外键信息
- 索引信息
- 用户权限
- 表之间的关系

```
Data Dictionary Example:
┌────────────┬────────────┬──────────┬───────────┬─────────┐
│ Table      │ Column     │ DataType │ Constraint│ Key     │
├────────────┼────────────┼──────────┼───────────┼─────────┤
│ STUDENTS   │ StudentID  │ VARCHAR  │ NOT NULL  │ PK      │
│ STUDENTS   │ Name       │ VARCHAR  │ NOT NULL  │         │
│ STUDENTS   │ ClassID    │ VARCHAR  │           │ FK→CLASS│
│ CLASSES    │ ClassID    │ VARCHAR  │ NOT NULL  │ PK      │
│ CLASSES    │ ClassName  │ VARCHAR  │           │         │
└────────────┴────────────┴──────────┴───────────┴─────────┘
```

### 2. Logical Schema（逻辑模式）

逻辑模式定义了数据库的**逻辑结构**——表有哪些、表之间的关系是什么、数据类型和约束是什么。

它是数据库的**蓝图（blueprint）**，就像建筑图纸一样。

### 3. 数据完整性（Data Integrity）

DBMS 通过**约束（Constraints）**确保数据的正确性：

| 约束类型 | 作用 | 例子 |
|:---|:---|:---|
| **NOT NULL** | 字段不能为空 | 学生姓名不能为空 |
| **UNIQUE** | 字段值必须唯一 | 邮箱地址不能重复 |
| **PRIMARY KEY** | 主键唯一且非空 | StudentID |
| **FOREIGN KEY** | 外键必须引用已存在的值 | ClassID 必须在 CLASSES 表中存在 |
| **CHECK** | 自定义条件 | Age BETWEEN 5 AND 100 |

### 4. 安全性与访问控制（Security & Access Control）

DBMS 提供多层安全保护：

**认证（Authentication）：** 验证用户身份（用户名 + 密码）

**授权（Authorization）：** 不同用户有不同权限：
- **DBA（Database Administrator）** — 最高权限，可以做任何事
- **开发者** — 可以创建/修改表结构
- **普通用户** — 只能查询指定的表
- **只读用户** — 只能 SELECT，不能修改

```
GRANT SELECT ON Students TO teacher_role;
GRANT SELECT, INSERT, UPDATE ON Grades TO teacher_role;
GRANT ALL ON ALL TABLES TO admin_role;
```

### 5. 并发访问控制（Concurrent Access Control）

当多个用户同时操作数据库时，可能出现问题：

> **类比：** 两个人同时编辑同一个 Google 文档的同一段话——谁的修改算数？

DBMS 使用**锁（Locking）**机制解决：
- **读锁（Shared Lock）：** 多人可以同时读，但不能改
- **写锁（Exclusive Lock）：** 一个人在改时，其他人既不能读也不能改

### 6. 备份与恢复（Backup & Recovery）

数据库需要定期备份，以防：
- 硬件故障（硬盘损坏）
- 软件错误（程序 bug 导致数据丢失）
- 人为错误（误删数据）
- 自然灾害（火灾、水灾）

恢复方法：
- **完全备份（Full Backup）** — 备份整个数据库
- **事务日志（Transaction Log）** — 记录每次操作，可以"回放"恢复

## 8.2.3 DBMS 工具 | DBMS Tools

### 开发者界面 / 查询构建器（Developer Interface / Query Builder）

提供图形化工具让用户构建查询，而不需要手写 SQL。

> **类比：** 就像手机上的"拖拽式"编程（如 Scratch），让你不用写代码也能实现功能。

### 查询处理器（Query Processor）

查询处理器是 DBMS 的"大脑"，负责：

1. **解析（Parsing）：** 检查 SQL 语句是否语法正确
2. **优化（Optimization）：** 找到执行查询的最佳方式
3. **执行（Execution）：** 实际访问数据并返回结果

> **为什么需要优化？**
>
> 同一个查询可以有多种执行方式。例如查询"所有北京的17岁学生"：
> - 方式A：先找所有北京学生，再从中筛选17岁的
> - 方式B：先找所有17岁学生，再从中筛选北京的
>
> 如果北京学生有1000人，17岁学生只有50人，那方式B更快！
> 查询处理器会自动选择最优方式。

## 8.2.4 DBMS vs 文件处理方法 | DBMS Advantages Summary

| 方面 | 文件处理方法 | DBMS |
|:---|:---|:---|
| **数据冗余** | 大量重复 | 最小化（通过规范化） |
| **数据一致性** | 难以保证 | 自动保证 |
| **数据共享** | 困难 | 多用户同时访问 |
| **数据完整性** | 无自动检查 | 约束自动检查 |
| **安全性** | 基本没有 | 精细的权限控制 |
| **并发控制** | 无 | 锁机制保证一致性 |
| **备份恢复** | 手动 | 自动化工具 |
| **数据独立性** | 无 | 程序与数据结构分离 |

### 数据独立性（Data Independence）

DBMS 提供了一个重要特性——**数据独立性**：
- **逻辑数据独立性：** 改变逻辑结构不影响应用程序
- **物理数据独立性：** 改变物理存储方式不影响逻辑结构

> **类比：** 你换了一个更大的书架（物理变化），但书的分类方式（逻辑结构）不变。你的借书习惯（应用程序）也不需要改变。

## 8.2.5 动手实践：用 Python sqlite3 创建数据库 | Hands-On with sqlite3

**SQLite** 是一个轻量级的关系数据库，它不需要安装服务器，数据库就是一个文件。Python 内置了 sqlite3 模块，可以直接使用。

> **为什么用 SQLite？**
> - 零配置，开箱即用
> - Python 内置支持
> - 非常适合学习 SQL
> - Android 和 iOS 应用广泛使用
> - 全球部署量超过 1 万亿！

让我们从零开始创建一个学校数据库：

In [ ]:
import sqlite3

# Create a database in memory
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create DEPARTMENTS table
cursor.execute(
    'CREATE TABLE Departments ('
    '    DeptID TEXT PRIMARY KEY,'
    '    DeptName TEXT NOT NULL,'
    '    Building TEXT'
    ')'
)

# Create TEACHERS table
cursor.execute(
    'CREATE TABLE Teachers ('
    '    TeacherID TEXT PRIMARY KEY,'
    '    Name TEXT NOT NULL,'
    '    DeptID TEXT,'
    '    FOREIGN KEY (DeptID) REFERENCES Departments(DeptID)'
    ')'
)

# Create STUDENTS table
cursor.execute(
    'CREATE TABLE Students ('
    '    StudentID TEXT PRIMARY KEY,'
    '    Name TEXT NOT NULL,'
    '    Age INTEGER,'
    '    Email TEXT UNIQUE'
    ')'
)

# Create COURSES table
cursor.execute(
    'CREATE TABLE Courses ('
    '    CourseID TEXT PRIMARY KEY,'
    '    CourseName TEXT NOT NULL,'
    '    Credits INTEGER,'
    '    TeacherID TEXT,'
    '    FOREIGN KEY (TeacherID) REFERENCES Teachers(TeacherID)'
    ')'
)

# Create ENROLLMENTS table (junction table for M:N relationship)
cursor.execute(
    'CREATE TABLE Enrollments ('
    '    StudentID TEXT,'
    '    CourseID TEXT,'
    '    Grade TEXT,'
    '    PRIMARY KEY (StudentID, CourseID),'
    '    FOREIGN KEY (StudentID) REFERENCES Students(StudentID),'
    '    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)'
    ')'
)

print('Database schema created successfully!')
print()

# Check what tables exist (query the data dictionary!)
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print('Tables in our database:')
for t in tables:
    print(f'  - {t[0]}')

conn.close()

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Recreate tables
cursor.execute('CREATE TABLE Departments (DeptID TEXT PRIMARY KEY, DeptName TEXT NOT NULL, Building TEXT)')
cursor.execute('CREATE TABLE Teachers (TeacherID TEXT PRIMARY KEY, Name TEXT NOT NULL, DeptID TEXT, FOREIGN KEY (DeptID) REFERENCES Departments(DeptID))')
cursor.execute('CREATE TABLE Students (StudentID TEXT PRIMARY KEY, Name TEXT NOT NULL, Age INTEGER, Email TEXT UNIQUE)')
cursor.execute('CREATE TABLE Courses (CourseID TEXT PRIMARY KEY, CourseName TEXT NOT NULL, Credits INTEGER, TeacherID TEXT, FOREIGN KEY (TeacherID) REFERENCES Teachers(TeacherID))')
cursor.execute('CREATE TABLE Enrollments (StudentID TEXT, CourseID TEXT, Grade TEXT, PRIMARY KEY (StudentID, CourseID), FOREIGN KEY (StudentID) REFERENCES Students(StudentID), FOREIGN KEY (CourseID) REFERENCES Courses(CourseID))')

# Insert sample data
cursor.executemany('INSERT INTO Departments VALUES (?, ?, ?)', [
    ('D01', 'Science', 'Building A'),
    ('D02', 'Humanities', 'Building B'),
])

cursor.executemany('INSERT INTO Teachers VALUES (?, ?, ?)', [
    ('T01', 'Mr. Wang', 'D01'),
    ('T02', 'Ms. Li', 'D01'),
    ('T03', 'Mr. Chen', 'D02'),
])

cursor.executemany('INSERT INTO Students VALUES (?, ?, ?, ?)', [
    ('S001', 'Alice', 17, 'alice@school.com'),
    ('S002', 'Bob', 16, 'bob@school.com'),
    ('S003', 'Carol', 17, 'carol@school.com'),
    ('S004', 'David', 16, 'david@school.com'),
    ('S005', 'Eve', 17, 'eve@school.com'),
])

cursor.executemany('INSERT INTO Courses VALUES (?, ?, ?, ?)', [
    ('CS101', 'Computer Science', 4, 'T01'),
    ('MA101', 'Mathematics', 4, 'T02'),
    ('EN101', 'English', 3, 'T03'),
])

cursor.executemany('INSERT INTO Enrollments VALUES (?, ?, ?)', [
    ('S001', 'CS101', 'A'),
    ('S001', 'MA101', 'A-'),
    ('S002', 'CS101', 'B+'),
    ('S002', 'EN101', 'A'),
    ('S003', 'MA101', 'B'),
    ('S003', 'EN101', 'A-'),
    ('S004', 'CS101', 'A'),
    ('S005', 'MA101', 'B+'),
    ('S005', 'EN101', 'A'),
])

conn.commit()
print('Sample data inserted successfully!')
print()

# Query 1: Show all students
print('=== All Students ===')
cursor.execute('SELECT * FROM Students')
for row in cursor.fetchall():
    print(f'  {row}')

print()
# Query 2: Show the data dictionary (table info)
print('=== Data Dictionary for Students Table ===')
cursor.execute('PRAGMA table_info(Students)')
print(f'{"ID":<4} {"Name":<12} {"Type":<10} {"NotNull":<8} {"PK":<4}')
for col in cursor.fetchall():
    print(f'{col[0]:<4} {col[1]:<12} {col[2]:<10} {col[3]:<8} {col[5]:<4}')

print()
# Query 3: Join query - which students take which courses?
print('=== Student Enrollments (JOIN query) ===')
cursor.execute(
    'SELECT s.Name, c.CourseName, e.Grade '
    'FROM Enrollments e '
    'JOIN Students s ON e.StudentID = s.StudentID '
    'JOIN Courses c ON e.CourseID = c.CourseID '
    'ORDER BY s.Name'
)
print(f'{"Student":<10} {"Course":<20} {"Grade":<6}')
for row in cursor.fetchall():
    print(f'{row[0]:<10} {row[1]:<20} {row[2]:<6}')

conn.close()

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create sample tables
cursor.execute('CREATE TABLE Students (StudentID TEXT PRIMARY KEY, Name TEXT NOT NULL, Age INTEGER CHECK(Age >= 5 AND Age <= 100))')
cursor.execute('CREATE TABLE Courses (CourseID TEXT PRIMARY KEY, CourseName TEXT NOT NULL UNIQUE)')

# Explore the data dictionary using sqlite_master
print('=== sqlite_master: The Data Dictionary ===')
print('This is where SQLite stores metadata about all database objects.')
print()
cursor.execute('SELECT type, name, sql FROM sqlite_master')
for row in cursor.fetchall():
    print(f'Type: {row[0]}')
    print(f'Name: {row[1]}')
    print(f'SQL:  {row[2]}')
    print()

print('The data dictionary tells us everything about the database structure!')
print('It is automatically maintained by the DBMS.')

# Demonstrate integrity constraint
print()
print('=== Data Integrity Demo ===')
cursor.execute('INSERT INTO Students VALUES (?, ?, ?)', ('S001', 'Alice', 17))
print('Inserted Alice (age 17): OK')

try:
    cursor.execute('INSERT INTO Students VALUES (?, ?, ?)', ('S001', 'Bob', 16))
except sqlite3.IntegrityError as e:
    print(f'Tried to insert Bob with same ID S001: BLOCKED! Error: {e}')
    print('Primary Key constraint prevents duplicate IDs!')

try:
    cursor.execute('INSERT INTO Students VALUES (?, ?, ?)', ('S002', None, 16))
except sqlite3.IntegrityError as e:
    print(f'Tried to insert student with NULL name: BLOCKED! Error: {e}')
    print('NOT NULL constraint prevents empty names!')

conn.close()

## 8.2.6 并发控制深入理解 | Understanding Concurrent Access

### 为什么并发控制很重要？

想象一个银行场景：

```
账户余额 = ¥1000

用户A（ATM取款 ¥500）：          用户B（网银转账 ¥300）：
1. 读取余额 = ¥1000              1. 读取余额 = ¥1000
2. 计算: 1000 - 500 = ¥500       2. 计算: 1000 - 300 = ¥700
3. 写入余额 = ¥500               3. 写入余额 = ¥700
```

如果没有并发控制，最终余额可能是 ¥500 或 ¥700，而不是正确的 ¥200！

这就是**丢失更新问题（Lost Update Problem）**。

### DBMS 如何解决？

1. **锁定（Locking）：** 操作数据前先"锁住"，操作完再"解锁"
2. **事务（Transaction）：** 一组操作要么全部成功，要么全部失败（ACID 原则）
3. **时间戳（Timestamping）：** 按时间顺序串行化操作

> **ACID 原则：**
> - **A**tomicity（原子性）：事务不可分割
> - **C**onsistency（一致性）：事务前后数据一致
> - **I**solation（隔离性）：事务之间互不干扰
> - **D**urability（持久性）：完成的事务永久保存

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('CREATE TABLE Accounts (AccID TEXT PRIMARY KEY, Owner TEXT, Balance REAL)')
cursor.execute('INSERT INTO Accounts VALUES (?, ?, ?)', ('A001', 'Alice', 1000.0))
cursor.execute('INSERT INTO Accounts VALUES (?, ?, ?)', ('A002', 'Bob', 500.0))
conn.commit()

print('=== Transaction Demo: Transfer 200 from Alice to Bob ===')
print()

# Show initial state
cursor.execute('SELECT * FROM Accounts')
print('Before transfer:')
for row in cursor.fetchall():
    print(f'  {row[0]}: {row[1]} - Balance: {row[2]}')

# Perform transfer as a transaction
try:
    cursor.execute('UPDATE Accounts SET Balance = Balance - 200 WHERE AccID = ?', ('A001',))
    cursor.execute('UPDATE Accounts SET Balance = Balance + 200 WHERE AccID = ?', ('A002',))
    conn.commit()  # Both succeed -> commit
    print('\nTransfer successful! Transaction committed.')
except Exception as e:
    conn.rollback()  # If any fails -> rollback ALL
    print(f'\nTransfer failed! Transaction rolled back. Error: {e}')

# Show final state
cursor.execute('SELECT * FROM Accounts')
print('\nAfter transfer:')
for row in cursor.fetchall():
    print(f'  {row[0]}: {row[1]} - Balance: {row[2]}')

print('\nNote: Both operations (debit Alice, credit Bob) succeed or fail TOGETHER.')
print('This is the Atomicity property of ACID.')

conn.close()

## 8.2.7 总结：为什么选择 DBMS？

### DBMS 的核心优势

1. **数据独立性（Data Independence）：** 应用程序不依赖于数据的物理存储方式
2. **减少冗余（Reduced Redundancy）：** 通过规范化，每条数据只存一次
3. **数据一致性（Data Consistency）：** 更新一处，所有引用自动更新
4. **数据完整性（Data Integrity）：** 约束自动检查数据的合法性
5. **安全性（Security）：** 精细的权限控制，谁能看什么、改什么
6. **并发控制（Concurrency Control）：** 多用户安全同时访问
7. **备份恢复（Backup & Recovery）：** 防止数据丢失
8. **标准化接口（Standard Interface）：** SQL 是通用的查询语言

> **考试重点：** 经常出现的题目是"State two advantages of using a DBMS over a file-based approach"。记住至少4-5个优势，并能用具体例子说明！

## 练习题 | Practice Exercises

### 练习 1：填空
1. DBMS 是 _______ 和物理数据库之间的软件层。
2. _______ 存储了关于数据库本身结构的元数据。
3. DBMS 使用 _______ 机制来处理多用户同时访问的问题。
4. ACID 中的 A 代表 _______，意思是事务不可分割。

### 练习 2：简答
1. 解释什么是"数据独立性"，并给出一个例子说明它的好处。
2. 描述"丢失更新问题"是如何发生的，DBMS 如何解决？
3. 列出 DBMS 提供的三种约束类型及其作用。

### 练习 3：对比分析
制作一个表格，比较文件处理方法和 DBMS 在以下方面的区别：
- 数据冗余
- 数据安全性
- 并发访问
- 数据完整性

### 练习 4：思考题
一家小型咖啡店只有一个收银员，每天记录不超过100条销售记录。店主在考虑是否需要使用 DBMS。请分析使用和不使用 DBMS 的利弊。

In [ ]:
print('=' * 60)
print('Exercise 1 Answers:')
print('=' * 60)
print('1. Users/Applications (用户/应用程序)')
print('2. Data Dictionary (数据字典)')
print('3. Locking (锁定)')
print('4. Atomicity (原子性)')
print()
print('=' * 60)
print('Exercise 2 Answers:')
print('=' * 60)
print()
print('1. Data Independence:')
print('   The ability to change the physical storage or logical')
print('   structure without affecting application programs.')
print('   Example: Moving the database to a faster disk does not')
print('   require changing any application code.')
print()
print('2. Lost Update Problem:')
print('   Two users read the same data simultaneously, both modify')
print('   it, and the second write overwrites the first.')
print('   DBMS solution: Use locking to ensure only one user can')
print('   modify the data at a time.')
print()
print('3. Three constraint types:')
print('   - NOT NULL: prevents empty values in required fields')
print('   - PRIMARY KEY: ensures unique identification of records')
print('   - FOREIGN KEY: ensures referential integrity between tables')
print()
print('=' * 60)
print('Exercise 4 Key Points:')
print('=' * 60)
print('For a small coffee shop:')
print('  Pro DBMS: data integrity, backup, future growth')
print('  Pro file: simpler, cheaper, sufficient for small scale')
print('  Recommendation: start with simple solution (e.g. SQLite),')
print('  which gives DBMS benefits without complexity overhead.')